# YOLO HAM10000 — تدريب وتشغيل على Colab

هذا الدفتر يحمّل المشروع من GitHub، ينزّل بيانات HAM10000، ثم يوفّر قسمين منفصلين:

1. **قسم التدريب:** إعادة تدريب النماذج المتوازنة  
2. **قسم التشغيل:** تحميل الأوزان الجاهزة وتشغيل استدلال / تقييم

**المستودع:** https://github.com/AlrgabeAh2025/yolo-testing-project

> شغّل Colab بـ **GPU**: `Runtime → Change runtime type → T4 GPU`.

## 0) الإعداد: استنساخ المشروع وتثبيت المتطلبات

In [ ]:
# إعدادات عامة — عدّل إن لزم
REPO_URL = "https://github.com/AlrgabeAh2025/yolo-testing-project.git"
PROJECT_DIR = "/content/yolo-testing-project"
USE_DRIVE = False  # True لحفظ النتائج على Google Drive
DRIVE_PROJECT = "/content/drive/MyDrive/yolo-testing-project"

import os, sys, subprocess, shutil
from pathlib import Path

def sh(cmd: str):
    print(">>", cmd)
    subprocess.run(cmd, shell=True, check=True)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = DRIVE_PROJECT

if not Path(PROJECT_DIR).exists():
    sh(f"git clone --depth 1 {REPO_URL} {PROJECT_DIR}")
else:
    print("Project exists:", PROJECT_DIR)
    # تحديث اختياري
    # sh(f"git -C {PROJECT_DIR} pull")

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(Path(PROJECT_DIR) / "scripts"))
print("CWD:", os.getcwd())

In [ ]:
# تثبيت الاعتماديات + التحقق من GPU
%pip -q install -U ultralytics opencv-python-headless pillow pandas tqdm PyYAML matplotlib seaborn scikit-learn

import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("تحذير: لا يوجد GPU. التدريب سيكون بطيئًا جدًا.")

## 1) تحميل البيانات وتحضير مجموعة YOLO

المسار الموصى به:
- ميتاداتا + أقنعة من Dataverse
- الصور من ISIC Archive API
- ثم `prepare_yolo_dataset.py --balance-train --malignant-ratio 1.2`

In [ ]:
from pathlib import Path

RAW = Path("data/raw")
META = RAW / "metadata" / "HAM10000_metadata.csv"
MASKS = RAW / "masks"
IMAGES = RAW / "images"
PROCESSED = Path("data/processed")

FORCE_REDOWNLOAD = False  # True لإعادة التنزيل
FORCE_REPREPARE = False   # True لإعادة بناء data/processed

need_meta = FORCE_REDOWNLOAD or not META.is_file()
need_masks = FORCE_REDOWNLOAD or not MASKS.is_dir() or not any(MASKS.glob("*"))
need_images = FORCE_REDOWNLOAD or not IMAGES.is_dir() or len(list(IMAGES.glob("*.jpg"))) < 10000

if need_meta or need_masks:
    # ميتاداتا + أقنعة فقط (بدون أرشيف الصور الثقيل)
    sh("python scripts/download_ham10000.py --source dataverse --skip-images")
else:
    print("Metadata/masks موجودة.")

if need_images:
    # تحميل الصور من ISIC (قابل للاستئناف)
    sh("python scripts/download_images_isic.py --workers 4")
else:
    n = len(list(IMAGES.glob("*.jpg")))
    print(f"Images موجودة: {n}")

need_prepare = (
    FORCE_REPREPARE
    or not (PROCESSED / "labels" / "train").is_dir()
    or not any((PROCESSED / "labels" / "train").glob("*.txt"))
)
if need_prepare:
    sh("python scripts/prepare_yolo_dataset.py --balance-train --malignant-ratio 1.2")
else:
    print("data/processed جاهزة.")

sh("python scripts/verify_dataset.py")

---
# الجزء أ — التدريب (Training)

شغّل خلايا هذا القسم فقط إذا أردت **إعادة تدريب** النماذج.

الوصفة المعتمدة: `configs/train_balanced.yaml`  
أسماء التشغيل: `yolov5su_balanced` / `yolov8s_balanced` / `yolov3-tinyu_balanced`

In [ ]:
# إعدادات التدريب
TRAIN_CONFIG = "configs/train_balanced.yaml"
EPOCHS = None          # None = استخدم القيمة من yaml (100). مثال: 30 للتجربة السريعة
BATCH = 16
WORKERS = 2

# اختر ما تريد تدريبه
TRAIN_MODELS = [
    ("yolov5su", "yolov5su_balanced"),
    ("yolov8s", "yolov8s_balanced"),
    ("yolov3-tinyu", "yolov3-tinyu_balanced"),
]

# True = أكمل من last.pt إن وُجد
RESUME_IF_EXISTS = True

print("سيتم تدريب:", [n for _, n in TRAIN_MODELS])

In [ ]:
from pathlib import Path

def train_one(model_key: str, run_name: str):
    last = Path("runs/detect") / run_name / "weights" / "last.pt"
    cmd = [
        "python", "scripts/train_yolo.py",
        "--model", model_key,
        "--config", TRAIN_CONFIG,
        "--name", run_name,
        "--batch", str(BATCH),
        "--workers", str(WORKERS),
        "--data", "configs/data.yaml",
    ]
    if EPOCHS is not None:
        cmd += ["--epochs", str(EPOCHS)]
    if RESUME_IF_EXISTS and last.is_file():
        cmd.append("--resume")
        print(f"Resume {run_name} from {last}")
    sh(" ".join(cmd))
    best = Path("runs/detect") / run_name / "weights" / "best.pt"
    print("best:", best, "exists=", best.is_file())
    return best

for model_key, run_name in TRAIN_MODELS:
    print("\n" + "=" * 60)
    print("Training", run_name)
    print("=" * 60)
    train_one(model_key, run_name)

print("\nانتهى التدريب.")

In [ ]:
# تقييم التعميم بعد التدريب (مستوى الصورة)
sh(
    "python scripts/test_generalization.py "
    "--models yolov5su_balanced yolov8s_balanced yolov3-tinyu_balanced "
    "--image-source processed"
)

---
# الجزء ب — التشغيل (Inference / Run)

شغّل هذا القسم لتشغيل نموذج جاهز **بدون إعادة تدريب**.

الأوزان الافتراضية موجودة في المستودع تحت `runs/detect/*/weights/best.pt`.

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt

CLASS_NAMES = {0: "benign", 1: "malignant"}

AVAILABLE_WEIGHTS = {
    "yolov5su_balanced": Path("runs/detect/yolov5su_balanced/weights/best.pt"),
    "yolov8s_balanced": Path("runs/detect/yolov8s_balanced/weights/best.pt"),
    "yolov3-tinyu_balanced": Path("runs/detect/yolov3-tinyu_balanced/weights/best.pt"),
}

# اختر النموذج للتشغيل
RUN_MODEL = "yolov5su_balanced"
CONF = 0.25
IOU = 0.5

weights = AVAILABLE_WEIGHTS[RUN_MODEL]
assert weights.is_file(), f"Missing weights: {weights}"
model = YOLO(str(weights))
print("Loaded:", weights)

In [ ]:
# تشغيل على صورة واحدة (اختَر مسار صورة من test أو ارفع صورة)
from pathlib import Path
import random

test_imgs = sorted(Path("data/processed/images/test").glob("*.jpg"))
if not test_imgs:
    test_imgs = sorted(Path("data/raw/images").glob("*.jpg"))

assert test_imgs, "لا توجد صور. شغّل قسم تحميل البيانات أولًا."

# غيّر IMAGE_PATH لمسار ثابت إن أردت
IMAGE_PATH = random.choice(test_imgs)
print("Image:", IMAGE_PATH)

results = model.predict(source=str(IMAGE_PATH), conf=CONF, iou=IOU, imgsz=640, verbose=False)
r0 = results[0]

if r0.boxes is None or len(r0.boxes) == 0:
    print("لا يوجد كشف فوق عتبة الثقة.")
else:
    confs = r0.boxes.conf.cpu().numpy()
    clss = r0.boxes.cls.cpu().numpy().astype(int)
    idx = int(confs.argmax())
    pred_cls = int(clss[idx])
    pred_conf = float(confs[idx])
    print(f"Prediction: {CLASS_NAMES.get(pred_cls, pred_cls)} | confidence={pred_conf:.3f}")

# عرض الصورة مع الصناديق
plotted = r0.plot()
plotted = cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(8, 8))
plt.imshow(plotted)
plt.axis("off")
plt.title(f"{RUN_MODEL} | {IMAGE_PATH.name}")
plt.show()

In [ ]:
# (اختياري) ارفع صورة من جهازك وشغّل عليها
UPLOAD_CUSTOM = False  # غيّر إلى True لرفع صورة

if UPLOAD_CUSTOM:
    from google.colab import files
    uploaded = files.upload()
    custom_path = next(iter(uploaded))
    results = model.predict(source=custom_path, conf=CONF, iou=IOU, imgsz=640, verbose=False)
    r0 = results[0]
    if r0.boxes is not None and len(r0.boxes):
        confs = r0.boxes.conf.cpu().numpy()
        clss = r0.boxes.cls.cpu().numpy().astype(int)
        idx = int(confs.argmax())
        print(CLASS_NAMES.get(int(clss[idx]), int(clss[idx])), float(confs[idx]))
    plotted = cv2.cvtColor(r0.plot(), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 8))
    plt.imshow(plotted)
    plt.axis("off")
    plt.show()
else:
    print("UPLOAD_CUSTOM=False — تخطي رفع صورة مخصصة.")

In [ ]:
# تقييم سريع لنموذج التشغيل على مجموعة الاختبار (بدون تدريب)
sh(
    f"python scripts/test_generalization.py --models {RUN_MODEL} "
    "--image-source processed"
)

---
## حفظ النتائج على Drive (اختياري)

In [ ]:
SAVE_TO_DRIVE = False
DRIVE_OUT = "/content/drive/MyDrive/yolo_runs_backup"

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)
    sh(f"rsync -a runs/detect/ {DRIVE_OUT}/detect/")
    sh(f"rsync -a runs/generalization/ {DRIVE_OUT}/generalization/")
    print("Saved to", DRIVE_OUT)
else:
    print("SAVE_TO_DRIVE=False")

---
### ملخص الاستخدام

| الهدف | ماذا تشغّل |
|---|---|
| إعداد المشروع + البيانات | خلايا القسم 0 و 1 |
| إعادة تدريب النماذج | خلايا **الجزء أ** |
| تشغيل نموذج جاهز على صورة | خلايا **الجزء ب** |
| تقييم فقط | خلية التقييم في الجزء ب |

أفضل نموذج حسب نتائج المشروع: `yolov5su_balanced`.